# **Physics Informed Neural Netwoks** (part 2)



---

## **Unsupervised Learning tasks**

In the previous lecture, we have seen that, given a specific differential equation along with a dataset $\{(x_j, y_j)\}_{j=1}^N$, it is possible to predict the solution of such differential equation $f$ with great accuracy. Nonetheless, all the physical constants were known a priori.

If such variables instead were unknown, we would perform an **Unsupervised Learning algorithm**: there is no label, or class to compare the prediction with.

In this case the training dataset of the function we want to fit is given, but we have to fit some parameters on which we have no information.

Therefore, tracking back the Stokes example, the $V$ and $\tau$ constants have to be predicted when performing the regression:

\begin{equation}
  \dot v + v/\tau - V = 0
\end{equation}

By such setting, we are more than performing a regression: we are defining the physical model the data obey to.

## **Hands on: Stokes' law with constant prediction**

**Goal**: fitting the damping curve for Stokes' law, this time predicting the $\tau$ and $V$ parameters along with the regression.

**Hints**

1.   As source code, take the file from the previous lecture.
2.   $V$ and $\tau$ are unknown variables. Instead of being passed as arguments for the constructor of the NN class, they can be set as random variables inside the constructor itself, according with the requires_grad=True flag:
```
self.tau = torch.rand(1, requires_grad=True)
self.V = torch.rand(1, requires_grad=True)
```
3.   In second place, the values for $\tau$ and $V$ need to be backpropagated in the process of training. To do so, simply append these values when instantiating the parameters of the Network:
```
params = list(self.parameters())
params.append(self.tau)
params.append(self.V)
```
where params are the python variables to be passed to the optimizer.



## **Schrödinger equation by PINN**

The next exercise will consist of simulating the Schrödinger equation by PINN: specifically, we will take the monodimensional harmonic oscillator as a hands-on. The equation we will focus on is the eigenvalue formulation of Schrödinger equation for an electron:

\begin{equation}
  H|\psi_n \rangle = E_n |\psi_n \rangle
\end{equation}

where $| \psi_n \rangle$ is an eigenstate for the $H$ Hamiltonian operator. The equation for the harmonic oscillator assumes the form

\begin{equation}
    -\frac{1}{2} \psi_n''(x) + \frac{(\omega x)^2}{2} \psi_n(x) = E_n \psi_n(x)
\end{equation}

## Atomic units

In the above equation, we set $\hbar=m_e=1$: we are dealing with Hartree atomic units (a.u.), where the above fundamental constants, plus $k_e$ and and $e$ (Coulomb constant and electronic charge) are set equal to $1$. In this frame, the characteristic distance of the system can be measured in terms of Bohr radius $a_0$:

\begin{equation}
  a_0 = \frac{\hbar^2}{e^2 m_e k_e} \rightarrow a_0 = 1
\end{equation}

When performing simulations of a quantum system, it is well convenient to adopt such measurement units: by doing so, we avoid the calculator to be overflown by infinitesimal digits, allowing memory and CPU to deal with numbers of scale ~$10^1$ (instead of $10^{-11}$, the length scale for Bohr radius in SI).

## **Hands on: monodimensional harmonic oscillator by PINN**

**Goal**: in the experiment we would like to deal with, the length of the system will be given by $6a_0$. Within this range, knowing the energy of the system, we would like to:

1.   Integrate the differential equation in order to shape the wavefunction $\psi(x)$
2.   Achieve the value for the $\omega$ pulsation: once such value is returned, get the $n$-th energy level by the formula $E_n = \omega (n + \frac{1}{2})$, as in a.u. $\hbar=1$. Set the energy of the ground state system to be $E_0=2.75$ a.u.



**Hints**


1.   Schrödinger equation is a second-order differential equation: in order to derive twice the Neural Network, it is possible to implement the following transformation:
```
psi_x = torch.autograd.grad(psi.sum(), x, create_graph=True)[0]
psi_xx = torch.autograd.grad(psi_x.sum(), x, create_graph=True)[0]
```
2.   As the range for the experiment to take place is $[-6a_0; 6a_0]$, in a.u. you may integrate the differential equation in the $[-6;6]$ domain.
3.   As for the NN architecture, try to implement 3 or 4 layers. The training will be hard: set at least 10.000 epochs.
4.   You may find out your dataset in SCHRODINGER_dataset.npz from the current folder.



## Digression: meaning of .sum() method in gradient evaluation

As we have seen from the code snippet above and the previous lectures, in order to derive the Neural Networks the .sum() is required on the output from the Feed-Forward model. Indeed, the output from the model is a vector-like object:

\begin{equation}
  f_{out} = [y_0, y_1, ..., y_n]
\end{equation}

In order to apply the gradient on the output tensor, avoiding to obtain a gradient matrix (but instead a vector), it is possible to sum over all the variables, thereafter apply the gradient:

\begin{equation}
  \nabla f_{out} = [\partial_0 y_0; \partial_1 y_1; ...; \partial_n y_n ]
\end{equation}

where $n$ is the upper bound for the batch size. This behaviour occurs as every $y_i$ depends on the $x_i$ variable; therefore $\partial_i y_j = \delta_{ij} y_j'$. Furthermore, it is possible to derive $f_{out}$ with respect to different variables, i.e. performing partial derivatives. In this case, we would collect an array of gradients, each on them containing any element derived by its specific variable.